# chat link

https://claude.ai/chat/9543cf72-db82-4c8b-85bc-234dae9c0dd8

# Import packages

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, sys
from pathlib import Path
for p in [Path.cwd()] + list(Path.cwd().parents):
    if p.name == 'Multifirefly-Project':
        os.chdir(p)
        sys.path.insert(0, str(p / 'multiff_analysis/multiff_code/methods'))
        break
    
from data_wrangling import specific_utils, process_monkey_information, general_utils
from pattern_discovery import pattern_by_trials, pattern_by_trials, cluster_analysis, organize_patterns_and_features
from visualization.matplotlib_tools import plot_behaviors_utils
from neural_data_analysis.neural_analysis_tools.get_neural_data import neural_data_processing
from neural_data_analysis.neural_analysis_tools.visualize_neural_data import plot_neural_data, plot_modeling_result
from neural_data_analysis.neural_analysis_tools.model_neural_data import transform_vars, neural_data_modeling, drop_high_corr_vars, drop_high_vif_vars
from neural_data_analysis.topic_based_neural_analysis.neural_vs_behavioral import prep_monkey_data, prep_target_data, neural_vs_behavioral_class
from neural_data_analysis.topic_based_neural_analysis.planning_and_neural import planning_and_neural_class, pn_utils, pn_helper_class, pn_aligned_by_seg, pn_aligned_by_event
from neural_data_analysis.topic_based_neural_analysis.planning_and_neural.pn_decoding.interactions import add_interactions, discrete_decoders
from neural_data_analysis.topic_based_neural_analysis.planning_and_neural.pn_decoding.interactions.band_conditioned import conditional_decoding_clf, conditional_decoding_reg, cond_decoding_plots
from neural_data_analysis.neural_analysis_tools.cca_methods import cca_class, cca_utils, cca_cv_utils
from neural_data_analysis.neural_analysis_tools.cca_methods.cca_plotting import cca_plotting, cca_plot_lag_vs_no_lag, cca_plot_cv
from machine_learning.ml_methods import regression_utils, regz_regression_utils, ml_methods_class, classification_utils, ml_plotting_utils, ml_methods_utils
from planning_analysis.show_planning import nxt_ff_utils, show_planning_utils
from neural_data_analysis.neural_analysis_tools.gpfa_methods import elephant_utils, fit_gpfa_utils, plot_gpfa_utils, gpfa_helper_class
from neural_data_analysis.neural_analysis_tools.align_trials import time_resolved_regression, time_resolved_gpfa_regression,plot_time_resolved_regression
from neural_data_analysis.neural_analysis_tools.align_trials import align_trial_utils
from neural_data_analysis.neural_analysis_tools.decoding_tools.general_decoding import cv_decoding
from neural_data_analysis.neural_analysis_tools.decoding_tools.decoding_helpers import decoding_model_specs
from neural_data_analysis.topic_based_neural_analysis.planning_and_neural.pn_decoding import pn_decoding_utils, plot_pn_decoding, decoding_by_num_ff
from neural_data_analysis.topic_based_neural_analysis.conditioned_responses import trajectory_clustering, plot_trajectories, neural_conditioned, neural_population, behavioral_decoding

import sys
import math
import gc
import subprocess
from pathlib import Path
from importlib import reload

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rc
from scipy import linalg, interpolate
from scipy.signal import fftconvolve
from scipy.io import loadmat
from scipy import sparse
import torch
from numpy import pi
from catboost import CatBoostRegressor

from sklearn.linear_model import RidgeCV
from sklearn.model_selection import cross_val_score

# Machine Learning imports
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.multivariate.cancorr import CanCorr

# Neuroscience specific imports
import neo
import rcca
import quantities as pq

plt.rcParams["animation.html"] = "html5"
os.environ['KMP_DUPLICATE_LIB_OK']='True'
rc('animation', html='jshtml')
matplotlib.rcParams.update(matplotlib.rcParamsDefault)
matplotlib.rcParams['animation.embed_limit'] = 2**128
pd.set_option('display.float_format', lambda x: '%.5f' % x)
np.set_printoptions(suppress=True)
print("done")

%load_ext autoreload
%autoreload 2

# retrieve data

In [ ]:
# raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Bruno/data_0312"
# raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Bruno/data_0330"
raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Bruno/data_0316"
# raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Bruno/data_0327"
# raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Bruno/data_0328"

In [ ]:
reduce_y_var_lags = False
planning_data_by_point_exists_ok = True
y_data_exists_ok = True
bin_width = 0.1

pn = pn_aligned_by_event.PlanningAndNeuralEventAligned(raw_data_folder_path=raw_data_folder_path, bin_width=bin_width)
pn.prep_data_to_analyze_planning(planning_data_by_point_exists_ok=planning_data_by_point_exists_ok)
pn.planning_data_by_point, cols_to_drop = general_utils.drop_columns_with_many_nans(
    pn.planning_data_by_point)
pn.get_x_and_y_data_for_modeling(exists_ok=y_data_exists_ok, reduce_y_var_lags=reduce_y_var_lags)

# I need end_at_stop_time = True for this
pn.prepare_seg_aligned_data(start_t_rel_event=-0.25, end_at_stop_time=True)

# trajectory clustering

In [ ]:
bin_size = 50
std_thresh=0.004
range_thresh=0.01
    
    
df = pn.rebinned_behav_data.copy()
df, seg_df, bounds = trajectory_clustering.preprocess_df(df)

X, seg_ids, entry_by_seg, df, stable_bounds, labels, bin_pairs = \
    trajectory_clustering.build_and_filter_stable_entries(df,
                                                          bin_size=bin_size,
                                                          std_thresh=std_thresh,
                                                          range_thresh=range_thresh)

plot_trajectories.plot_clusters_grid_subplots(
    df, seg_ids, labels, bounds, align=False, show_grid=True, entry_by_seg=entry_by_seg
)
plot_trajectories.plot_clusters(
    df, seg_ids, labels, bounds=stable_bounds,
    align=False, show_grid=True, entry_by_seg=entry_by_seg,
)

In [ ]:
# both thresholds
plot_trajectories.plot_curvature_diagnostics(df, window=5, std_thresh=0.002, range_thresh=0.01)

# # no thresholds — just see the raw distributions
# plot_trajectories.plot_curvature_diagnostics(df, window=5)

In [ ]:
# find seg in an area

plot_trajectories. find_segs_in_bin(entry_by_seg, seg_ids, labels, bin_idx=0,
                 x_range=(-80, -60), y_range=(85, 135))

In [ ]:
stop

# conditioned variance

In [ ]:
# get stable-phase entry time for each seg_id
entry_times = {}
for seg_id, entry_xy in entry_by_seg.items():
    seg_df = df[df["new_segment"] == seg_id].sort_values("bin_in_new_seg")
    ex, ey = entry_xy
    dists = (seg_df["cur_ff_rel_x"] - ex).abs() + (seg_df["cur_ff_rel_y"] - ey).abs()
    entry_times[seg_id] = seg_df.loc[dists.idxmin(), "t_center"]

In [ ]:
reload(neural_conditioned)

In [ ]:
n_bins = 20
pre = 0.2
t_bins, psth, residuals, trial_rates = neural_conditioned.compute_psth(
    pn.spikes_df, entry_times, seg_ids, labels, df, n_bins=n_bins, pre=pre
)

for (lbl, nid), dr in residuals.items():
    var = (dr**2).mean()
    print(f"bin {lbl}, neuron {nid}: residual var = {var:.2f} Hz²")
    
fig = neural_conditioned.plot_summary(t_bins, psth, residuals, trial_rates, seg_ids)
plt.show()

## after compute_psth

In [ ]:
# # after compute_psth:
# t_bins, psth, residuals, trial_rates = neural_conditioned.compute_psth(...)

neuron_id = 1

# for one neuron, timecourse
eta2_t = neural_conditioned.compute_eta2_timecourse(psth, residuals, neuron_id, t_bins)

# for one neuron, scalar (more robust — averages over time first, then ANOVA)
eta2 = neural_conditioned.compute_eta2_scalar(psth, residuals, trial_rates, seg_ids, neuron_id)

# for all neurons at once — already produces the full summary figure
fig = neural_conditioned.plot_summary(t_bins, psth, residuals, trial_rates, seg_ids)

# compute duration per seg_id

In [ ]:
# compute duration per seg_id
durations = {}
for seg_id in seg_ids:
    seg_df = df[df["new_segment"] == seg_id]
    t0 = entry_times[seg_id]
    t_end = seg_df["t_center"].max()
    durations[seg_id] = t_end - t0

# check if duration differs across bins
import pandas as pd
dur_df = pd.DataFrame({
    "seg_id": seg_ids,
    "label": labels,
    "duration": [durations[s] for s in seg_ids]
})
print(dur_df.groupby("label")["duration"].describe())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_duration_by_bin(dur_df):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # left: boxplot of duration per bin
    axes[0].boxplot(
        [dur_df[dur_df["label"] == lbl]["duration"].values
         for lbl in sorted(dur_df["label"].unique())],
        labels=sorted(dur_df["label"].unique())
    )
    axes[0].set_xlabel("bin label")
    axes[0].set_ylabel("duration (s)")
    axes[0].set_title("trajectory duration per bin")

    # right: scatter of duration vs bin with jitter
    for lbl in sorted(dur_df["label"].unique()):
        sub = dur_df[dur_df["label"] == lbl]
        jitter = np.random.uniform(-0.2, 0.2, len(sub))
        axes[1].scatter(lbl + jitter, sub["duration"].values,
                        alpha=0.4, s=15)
    axes[1].set_xlabel("bin label")
    axes[1].set_ylabel("duration (s)")
    axes[1].set_title("duration scatter per bin")

    plt.tight_layout()
    return fig

fig = plot_duration_by_bin(dur_df)
plt.show()

# population

In [ ]:
fig = neural_population.plot_population_analysis(
    psth, trial_rates, seg_ids, labels, t_bins, method="lda"
)
plt.show()

# behavioral decoding

In [ ]:
fig, results = behavioral_decoding.run_all(
    psth, residuals, trial_rates, seg_ids, labels, entry_by_seg,
    entry_times, df, t_bins, n_bins=3, method="lda"
)
plt.show()

# inspect traj

In [ ]:
stop!

In [ ]:
# grid view — see all at once, seg_id in title, entry coords annotated in red
bin_idx = 2

print(f"std_thresh={std_thresh}, range_thresh={range_thresh}")

plot_trajectories.inspect_bin_grid(df, seg_ids, labels, entry_by_seg, bin_idx=bin_idx, ncols=3,
                                   std_thresh=std_thresh, range_thresh=range_thresh)

# # or step through one at a time with curvature trace
# plot_trajectories.inspect_bin_one_by_one(df, seg_ids, labels, entry_by_seg, bin_idx=bin_idx,
#                                           std_thresh=std_thresh, range_thresh=range_thresh)

In [ ]:
df.loc[df['new_segment'] == 28, ['curv_of_traj', 'cur_ff_rel_x', 'cur_ff_rel_y']]

# iterate over std_thresh

In [ ]:
for std_thresh in [0.00005, 0.0001, 0.0002, 0.0004, 0.001, 0.002]:
    print(f"\n--- std_thresh={std_thresh} ---")

    df = pn.rebinned_behav_data.copy()
    df, seg_df, bounds = trajectory_clustering.preprocess_df(df)

    X, seg_ids, entry_by_seg, df, stable_bounds, labels, bin_pairs = \
        trajectory_clustering.build_and_filter_stable_entries(df, std_thresh=std_thresh)

    if len(X) == 0:
        print("  No trajectories remaining, skipping plots.")
        continue

    plot_trajectories.plot_clusters_grid_subplots(
        df, seg_ids, labels, bounds, align=False, show_grid=True
    )
    plot_trajectories.plot_clusters(
        df, seg_ids, labels, bounds=stable_bounds,
        align=False, show_grid=True, entry_by_seg=entry_by_seg,
    )

# loop over sessions

In [ ]:
reload(behavioral_decoding)

In [ ]:
MONKEY_NAME   = 'monkey_Bruno'
RAW_DATA_DIR  = 'all_monkey_data/raw_monkey_data'

all_results = behavioral_decoding.run_decoding_for_monkey(
    monkey_name=MONKEY_NAME,
    raw_data_dir=RAW_DATA_DIR,
    load_if_exists=True,
)



In [ ]:
summary_df = behavioral_decoding.get_summary_df(
    all_results=all_results,
    monkey_name=MONKEY_NAME,
    load_if_exists=True,
)

behavioral_decoding.print_aggregate_stats(summary_df)
fig_summary = behavioral_decoding.plot_summary(summary_df, MONKEY_NAME)
